# Media Format Converter - One-Click Version

Convert MP4 files to MP3 format with a single click!

## How to use:
1. Run the cell below (click the play button or press Shift+Enter)
2. Wait for dependencies to install
3. Upload your MP4 files when prompted
4. Wait for conversion to complete
5. Download your MP3 files automatically

That's it! Everything runs automatically.

In [ ]:
# ===================================================================
# Media Format Converter - One-Click Version
# ===================================================================

import sys
import subprocess
import os
import glob
from pathlib import Path

# ===================================================================
# STEP 1: Install Dependencies
# ===================================================================

print("\n" + "=" * 70)
print("STEP 1/5: Installing Dependencies")
print("=" * 70)

# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Running in local environment")

# Install ffmpeg
print("\nInstalling ffmpeg...")
subprocess.run(['apt-get', 'update'], capture_output=True, check=True)
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True, check=True)

# Verify ffmpeg
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode == 0:
    version_line = result.stdout.split('\n')[0]
    print(f"✓ {version_line}")
else:
    print("✗ ffmpeg installation failed")
    sys.exit(1)

# Install Python packages
print("\nInstalling Python packages...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ffmpeg-python'], 
               capture_output=True, check=True)
print("✓ ffmpeg-python installed successfully")

import ffmpeg
from google.colab import files

print("\n✓ All dependencies installed successfully!")

# ===================================================================
# STEP 2: Define Converter Functions
# ===================================================================

print("\n" + "=" * 70)
print("STEP 2/5: Loading Converter Functions")
print("=" * 70)

def convert_mp4_to_mp3(input_file, output_file=None):
    """Convert MP4 file to MP3 format"""
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"File not found: {input_file}")
    
    if not input_file.lower().endswith('.mp4'):
        raise ValueError("Input file must be MP4 format")
    
    # Generate output filename
    if output_file is None:
        output_file = str(Path(input_file).with_suffix('.mp3'))
    elif not output_file.lower().endswith('.mp3'):
        output_file += '.mp3'
    
    try:
        print(f"  Converting: {input_file}")
        stream = ffmpeg.input(input_file)
        stream = ffmpeg.output(stream, output_file, acodec='libmp3lame', audio_bitrate='192k')
        ffmpeg.run(stream, overwrite_output=True, quiet=True)
        print(f"  ✓ Completed: {output_file}")
        return output_file
    except Exception as e:
        print(f"  ✗ Failed: {input_file} - {str(e)}")
        return None

print("✓ Converter functions loaded")

# ===================================================================
# STEP 3: Upload MP4 Files
# ===================================================================

print("\n" + "=" * 70)
print("STEP 3/5: Upload MP4 Files")
print("=" * 70)
print("\nPlease select your MP4 file(s) to upload...\n")

uploaded = files.upload()

if not uploaded:
    print("\n✗ No files uploaded. Exiting.")
    sys.exit(0)

print(f"\n✓ Uploaded {len(uploaded)} file(s):")
for filename in uploaded.keys():
    size_mb = len(uploaded[filename]) / (1024 * 1024)
    print(f"  - {filename} ({size_mb:.2f} MB)")

# ===================================================================
# STEP 4: Convert All Files
# ===================================================================

print("\n" + "=" * 70)
print("STEP 4/5: Converting Files")
print("=" * 70)

# Find all MP4 files
mp4_files = [f for f in uploaded.keys() if f.lower().endswith('.mp4')]

if not mp4_files:
    print("\n✗ No MP4 files found in uploaded files.")
    sys.exit(0)

print(f"\nFound {len(mp4_files)} MP4 file(s) to convert\n")

success_count = 0
fail_count = 0
converted_files = []

for i, mp4_file in enumerate(mp4_files, 1):
    print(f"[{i}/{len(mp4_files)}] Processing: {mp4_file}")
    result = convert_mp4_to_mp3(mp4_file)
    if result:
        success_count += 1
        converted_files.append(result)
    else:
        fail_count += 1
    print()

print("=" * 70)
print("Conversion Summary:")
print(f"  ✓ Success: {success_count} file(s)")
print(f"  ✗ Failed: {fail_count} file(s)")
print("=" * 70)

# ===================================================================
# STEP 5: Download MP3 Files
# ===================================================================

if converted_files:
    print("\n" + "=" * 70)
    print("STEP 5/5: Downloading MP3 Files")
    print("=" * 70)
    print(f"\nDownloading {len(converted_files)} file(s)...\n")
    
    for mp3_file in converted_files:
        print(f"  Downloading: {mp3_file}")
        files.download(mp3_file)
    
    print("\n" + "=" * 70)
    print("✓ ALL DONE!")
    print("=" * 70)
    print(f"\nSuccessfully converted and downloaded {len(converted_files)} file(s)!")
    print("Check your Downloads folder for the MP3 files.\n")
else:
    print("\n✗ No files were successfully converted.")

# ===================================================================
# Optional: Cleanup
# ===================================================================

print("\nCleaning up temporary files...")
for file in glob.glob("*.mp4") + glob.glob("*.mp3"):
    try:
        os.remove(file)
    except:
        pass
print("✓ Cleanup completed\n")